# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmoeed1090/Fly-rank-starter-assignment/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Two paper findings + my methodology questions

The first finding I focus on is that FlyRank's workflow uses observable content-performance signals to identify pages that may deserve editorial attention. The important methodological question is how the outcome or label is defined. If the label is derived from future performance, the validation must ensure that future information is not available in the features used at prediction time.

My question is whether the validation design fully preserves this temporal separation. A model can appear useful if observations from the future are allowed to influence training or feature construction, even indirectly.

The second finding is that the system is intended to support editorial prioritization rather than replace human judgment. The methodology should therefore evaluate whether the model improves the ranking of useful review candidates, rather than treating model complexity or raw accuracy as the goal.

My question is whether the evaluation metric matches the actual decision. Since the intended output is a ranked review queue, ranking metrics such as Average Precision and Precision@K are more informative than accuracy alone.

Overall, these questions do not reject the paper's approach. They identify the validation assumptions that need to be checked before treating the results as reliable decision-support evidence.

In [1]:
# ML-09 is intentionally self-contained.
# Load the same FlyRank warehouse used in the previous notebooks.

import os
import json
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score
)

print("Imports completed.")

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

df = dataset["train"].to_pandas()

# Keep the same working-size approach used in the previous notebook.
df = df.head(50000).copy()

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

Imports completed.


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 21.6MB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset shape: (50000, 30)
Columns: 30


### 2. My model under an honest split

The model is evaluated using a time-aware split because the target represents a future outcome.

The current-day features are used to predict whether the same page experiences a measurable decline on its next observed calendar day.

The final 20% of usable observations by date are reserved for testing. Earlier observations are used for training.

This design prevents later observations from being randomly mixed into the training data when evaluating earlier observations.

The model uses only current-day search-performance information:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ctr`

The target is an observed future-performance proxy, not a direct label saying that a page needs a refresh.

In [2]:
# ---------------------------------------------------------
# Prepare dates and current-day features
# ---------------------------------------------------------

df["report_date"] = pd.to_datetime(
    df["report_date"],
    errors="coerce"
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

for col in feature_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0)

# Current-day CTR
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, 1)
)

df["ctr"] = df["ctr"].clip(0, 1)

print("Current-day features prepared.")

display(
    df[
        feature_cols + ["ctr"]
    ].describe()
)

Current-day features prepared.


,gsc_impressions,gsc_clicks,gsc_avg_position,ctr
count,50000.000000,50000.000000,50000.000000,50000.000000
mean,15.461200,0.105300,28.559778,0.007629
std,25.695012,0.451637,22.826635,0.048061
min,1.000000,0.000000,0.000000,0.000000
25%,3.000000,0.000000,9.000000,0.000000
50%,8.000000,0.000000,21.666667,0.000000
75%,18.000000,0.000000,43.000000,0.000000
max,818.000000,16.000000,141.000000,1.000000


In [3]:
# ---------------------------------------------------------
# Construct the future outcome
# ---------------------------------------------------------

# Sort by page and date.
# client_hash_id and content_hash_id are used ONLY for grouping.
df = df.sort_values(
    [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ]
).reset_index(drop=True)

group_cols = [
    "client_hash_id",
    "content_hash_id"
]

# Next observed date for the same page
df["future_date"] = (
    df.groupby(group_cols)["report_date"]
      .shift(-1)
)

# Next day's impressions
df["future_impressions"] = (
    df.groupby(group_cols)["gsc_impressions"]
      .shift(-1)
)

# Require an immediately following calendar day.
valid_future = (
    df["future_date"]
    ==
    df["report_date"] + pd.Timedelta(days=1)
)

# Start with missing target.
df["future_decline"] = np.nan

# Only evaluate pages with at least 5 current impressions.
eligible = (
    valid_future
    &
    (df["gsc_impressions"] >= 5)
)

# Future decline = next-day impressions are at least 20%
# lower than the current-day impressions.
df.loc[eligible, "future_decline"] = (
    df.loc[eligible, "future_impressions"]
    <=
    df.loc[eligible, "gsc_impressions"] * 0.80
).astype(int)

print(
    "Rows with measurable future outcome:",
    df["future_decline"].notna().sum()
)

print("\nFuture outcome distribution:")
display(
    df["future_decline"]
    .value_counts(dropna=False)
)

Rows with measurable future outcome: 27422

Future outcome distribution:


,count
future_decline,
NaN,22578
0.0,17566
1.0,9856


In [4]:
# ---------------------------------------------------------
# Create honest time-aware train/test split
# ---------------------------------------------------------

model_df = df[
    df["future_decline"].notna()
].copy()

model_df["future_decline"] = (
    model_df["future_decline"].astype(int)
)

model_df = model_df.sort_values(
    "report_date"
).reset_index(drop=True)

split_index = int(
    len(model_df) * 0.80
)

train_df = model_df.iloc[
    :split_index
].copy()

test_df = model_df.iloc[
    split_index:
].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "\nTrain period:",
    train_df["report_date"].min(),
    "to",
    train_df["report_date"].max()
)

print(
    "Test period:",
    test_df["report_date"].min(),
    "to",
    test_df["report_date"].max()
)

print(
    "\nTrain future-decline rate:",
    round(train_df["future_decline"].mean(), 4)
)

print(
    "Test future-decline rate:",
    round(test_df["future_decline"].mean(), 4)
)

Train rows: 21937
Test rows: 5485

Train period: 2025-01-27 00:00:00 to 2025-02-24 00:00:00
Test period: 2025-02-24 00:00:00 to 2025-02-26 00:00:00

Train future-decline rate: 0.3728
Test future-decline rate: 0.3059


In [5]:
# ---------------------------------------------------------
# Train the same Random Forest approach
# ---------------------------------------------------------

model_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr"
]

X_train = train_df[
    model_features
]

y_train = train_df[
    "future_decline"
]

X_test = test_df[
    model_features
]

y_test = test_df[
    "future_decline"
]

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(
    X_train,
    y_train
)

model_probability = model.predict_proba(
    X_test
)[:, 1]

model_prediction = (
    model_probability >= 0.50
).astype(int)

print("Random Forest trained.")

Random Forest trained.


In [6]:
# ---------------------------------------------------------
# Week-4 baseline recreated independently
# ---------------------------------------------------------

baseline_test = test_df.copy()

baseline_test["baseline_score"] = (
    baseline_test["gsc_impressions"] * 0.4
    +
    (1 - baseline_test["ctr"]) * 40
    +
    baseline_test["gsc_avg_position"] * 0.2
)

# Model ranking
baseline_test["model_probability"] = (
    model_probability
)

# ---------------------------------------------------------
# Evaluation helpers
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k=20):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["y"].mean()


# Baseline metrics
baseline_auc = roc_auc_score(
    y_test,
    baseline_test["baseline_score"]
)

baseline_ap = average_precision_score(
    y_test,
    baseline_test["baseline_score"]
)

baseline_p20 = precision_at_k(
    y_test,
    baseline_test["baseline_score"],
    20
)


# Model metrics
model_auc = roc_auc_score(
    y_test,
    model_probability
)

model_ap = average_precision_score(
    y_test,
    model_probability
)

model_p20 = precision_at_k(
    y_test,
    model_probability,
    20
)

base_rate = y_test.mean()

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "ROC_AUC": [
        baseline_auc,
        model_auc
    ],
    "Average_Precision": [
        baseline_ap,
        model_ap
    ],
    "Precision_at_20": [
        baseline_p20,
        model_p20
    ]
})

print(
    f"Test future-decline base rate: "
    f"{base_rate:.4f} ({base_rate * 100:.2f}%)"
)

display(comparison)

Test future-decline base rate: 0.3059 (30.59%)


,Method,ROC_AUC,Average_Precision,Precision_at_20
0,Week-4 Baseline,0.465994,0.286454,0.35
1,Random Forest,0.510678,0.313743,0.25


### Before / after interpretation

The Week-4 baseline is the transparent rule-based reference point. The Random Forest is compared against it using the same test observations and the same future-outcome definition.

The comparison should not be interpreted as proof that the model is better simply because it is more complex. The relevant question is whether the learned model provides better discrimination or ranking performance on the same held-out future period.

Precision@20 is particularly relevant because the intended use is a small human-review queue. The test-set base rate is reported alongside the metric so that the precision result has context.

## 3. Leakage audit

The final feature set contains only current-day search-performance information.

The target is constructed from the following day's impressions, so future fields are deliberately excluded from the model features.

Identifiers are used only to group observations belonging to the same page. They are not supplied to the model as predictive features.

The following potential leakage sources were checked:

- `future_impressions`: excluded because it is future information.
- `future_date`: excluded because it identifies the future observation.
- `future_decline`: excluded because it is the target.
- `client_hash_id`: grouping identifier only.
- `content_hash_id`: grouping identifier only.
- `report_date`: used for temporal ordering and splitting, not as a model feature.
- Week-4 `baseline_score`: excluded because it is an output of the previous rule.
- `reason_code`: excluded because it is a rule-generated output.
- `action`: excluded because it is a downstream decision label.

The audit therefore checks both future leakage and feature contamination from the previous baseline.

In [7]:
# ---------------------------------------------------------
# Leakage audit
# ---------------------------------------------------------

forbidden_features = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "future_date",
    "future_impressions",
    "future_decline",
    "baseline_score",
    "reason_code",
    "action"
]

print("Final model features:")
print(model_features)

print("\nPotential leakage fields:")
print(forbidden_features)

leakage_in_features = [
    col
    for col in model_features
    if col in forbidden_features
]

print(
    "\nLeakage fields accidentally included:",
    leakage_in_features
)

assert len(leakage_in_features) == 0

print(
    "\nPASS: no explicitly identified leakage field "
    "is included in the final feature vector."
)

Final model features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr']

Potential leakage fields:
['client_hash_id', 'content_hash_id', 'report_date', 'future_date', 'future_impressions', 'future_decline', 'baseline_score', 'reason_code', 'action']

Leakage fields accidentally included: []

PASS: no explicitly identified leakage field is included in the final feature vector.


In [8]:
# ---------------------------------------------------------
# Verify that model features are current-day fields
# ---------------------------------------------------------

print("Feature availability check:")

availability_check = pd.DataFrame({
    "feature": model_features,
    "current_day_information": [
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ],
    "future_information": [
        "No",
        "No",
        "No",
        "No"
    ]
})

display(availability_check)

Feature availability check:


,feature,current_day_information,future_information
0,gsc_impressions,Yes,No
1,gsc_clicks,Yes,No
2,gsc_avg_position,Yes,No
3,ctr,Yes,No


In [9]:
# ---------------------------------------------------------
# Feature importance for the final model
# ---------------------------------------------------------

importance_df = pd.DataFrame({
    "feature": model_features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

display(importance_df)

,feature,importance
0,gsc_avg_position,0.533926
1,gsc_impressions,0.322764
2,ctr,0.110217
3,gsc_clicks,0.033093


## 4. Claim rewrite

### Original-style claim

The Random Forest predicts which pages need a content refresh.

### Safer claim

On the held-out time period, the Random Forest provided a decision-support ranking for pages associated with the defined next-day impression-decline proxy. Its performance was measured against the Week-4 rule-based baseline using the same test observations and metrics.

The result should be treated as directional evidence about the usefulness of the current-day search signals for prioritization. It does not establish that the model causes better content decisions, nor does the future-decline proxy prove that a page actually needs a refresh.

The evaluation is therefore **observed and measured**, while the practical recommendation remains **decision-support** rather than an automatic editorial decision.

In [10]:
# ---------------------------------------------------------
# Final validation receipt
# ---------------------------------------------------------

validation_receipt = {
    "random_seed": 42,
    "test_base_rate": float(base_rate),
    "baseline_roc_auc": float(baseline_auc),
    "model_roc_auc": float(model_auc),
    "baseline_average_precision": float(baseline_ap),
    "model_average_precision": float(model_ap),
    "baseline_precision_at_20": float(baseline_p20),
    "model_precision_at_20": float(model_p20),
    "model_features": model_features,
    "split": "time-aware 80/20",
    "future_decline_definition": (
        "next calendar day impressions <= 80% of current impressions "
        "for rows with at least 5 current impressions"
    )
}

os.makedirs(
    "work/outputs",
    exist_ok=True
)

with open(
    "work/outputs/ml09_validation_receipt.json",
    "w"
) as f:
    json.dump(
        validation_receipt,
        f,
        indent=2
    )

print("ML-09 validation receipt saved.")

print("\nFinal metrics:")
print(
    f"Baseline ROC-AUC: {baseline_auc:.4f}"
)
print(
    f"Model ROC-AUC:    {model_auc:.4f}"
)
print(
    f"Baseline AP:      {baseline_ap:.4f}"
)
print(
    f"Model AP:         {model_ap:.4f}"
)
print(
    f"Baseline P@20:    {baseline_p20:.4f}"
)
print(
    f"Model P@20:       {model_p20:.4f}"
)

ML-09 validation receipt saved.

Final metrics:
Baseline ROC-AUC: 0.4660
Model ROC-AUC:    0.5107
Baseline AP:      0.2865
Model AP:         0.3137
Baseline P@20:    0.3500
Model P@20:       0.2500


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.